In [9]:
import pandas as pd
import numpy as np

In [10]:
df = pd.read_csv("insomnia_dataset.csv")

In [11]:
df.head()

,email_id,date,age,gender,content_type_primary,usage_time_10_2,num_sessions_night,max_continuous_usage,reels_usage_time,phone_pickups_after_10,...,cannot_stop_scrolling,unplanned_usage,sleep_time,wake_up_time,daily_stress_level,overthinking_at_night,sleep_latency,night_awakenings,sleep_quality,insomnia_label
0,priyashah768@gmail.com,2026-03-15,18,female,Educational/Entertainment,01:01,12,00:38,00:25,25,...,1,1,23:47,06:29,3,1,00:08,0,4,0
1,karangupta@gmail.com,2026-03-15,20,female,Educational,02:39,11,01:32,00:53,16,...,1,1,22:02,06:53,3,3,00:45,2,2,1
2,vikasshah871@gmail.com,2026-03-15,21,female,Entertainment,02:40,10,00:37,00:34,5,...,4,4,00:44,07:45,5,5,00:57,2,2,1
3,karanreddy6@gmail.com,2026-03-15,24,male,Entertainment,02:51,15,02:39,02:00,17,...,4,4,23:33,05:49,4,5,00:59,4,2,1
4,vikassharma@gmail.com,2026-03-15,30,male,Motivational,00:35,13,00:11,00:11,22,...,4,3,23:54,06:18,2,1,00:13,0,5,0


In [12]:
df.columns

Index(['email_id', 'date', 'age', 'gender', 'content_type_primary',
       'usage_time_10_2', 'num_sessions_night', 'max_continuous_usage',
       'reels_usage_time', 'phone_pickups_after_10', 'lose_track_of_time',
       'cannot_stop_scrolling', 'unplanned_usage', 'sleep_time',
       'wake_up_time', 'daily_stress_level', 'overthinking_at_night',
       'sleep_latency', 'night_awakenings', 'sleep_quality', 'insomnia_label'],
      dtype='object')

In [13]:
df.dtypes

email_id                  object
date                      object
age                        int64
gender                    object
content_type_primary      object
usage_time_10_2           object
num_sessions_night         int64
max_continuous_usage      object
reels_usage_time          object
phone_pickups_after_10     int64
lose_track_of_time         int64
cannot_stop_scrolling      int64
unplanned_usage            int64
sleep_time                object
wake_up_time              object
daily_stress_level         int64
overthinking_at_night      int64
sleep_latency             object
night_awakenings           int64
sleep_quality              int64
insomnia_label             int64
dtype: object


# **DATA CLEANING AND TRANSFORMATION**


In [14]:
df['gender'].value_counts()

gender
female    616
male      532
Name: count, dtype: int64

In [15]:
d1 = {"female":1 , "male":2}
df['gender'] = df['gender'].map(d1)
df['gender'].value_counts()

gender
1    616
2    532
Name: count, dtype: int64

In [16]:
df['content_type_primary'].value_counts()

content_type_primary
Entertainment                309
Educational                  230
Educational/Entertainment    204
Motivational                 176
Mixed                        141
Emotional/Drama               88
Name: count, dtype: int64

In [17]:
d2 = {"Entertainment":1 , "Educational/Entertainment": 4, "Educational":2, "Motivational":1, "Emotional/Drama":5, "Mixed": 3}
df['content_type_primary'] = df['content_type_primary'].map(d2)
df['content_type_primary'].value_counts()

content_type_primary
1    485
2    230
4    204
3    141
5     88
Name: count, dtype: int64

In [18]:
def convert_to_minutes(column):
    return column.str.split(':').apply(lambda x: int(x[0]) * 60 + int(x[1]))


In [19]:
df['usage_time_10_2'] = convert_to_minutes(df['usage_time_10_2'])
df['max_continuous_usage'] = convert_to_minutes(df['max_continuous_usage'])
df['reels_usage_time'] = convert_to_minutes(df['reels_usage_time'])
df['sleep_latency'] = convert_to_minutes(df['sleep_latency'])


## creating sleeping minutes from wakeup time and sleep time

In [20]:
def time_diff_handle_midnight(start_col, end_col):
    start = pd.to_datetime(start_col, format='%H:%M')
    end = pd.to_datetime(end_col, format='%H:%M')

    diff = (end - start).dt.total_seconds() / 60
    diff = diff.apply(lambda x: x if x >= 0 else x + 1440)  # add 24h if negative

    return diff.astype(int)

In [21]:
df['sleep_duration'] = time_diff_handle_midnight(df['sleep_time'] , df['wake_up_time'])

In [22]:
df['sleep_duration']

0       402
1       531
2       421
3       376
4       384
       ... 
1143    443
1144    416
1145    349
1146    345
1147    381
Name: sleep_duration, Length: 1148, dtype: int64




#**LOGISTIC REGRESSION**



In [23]:

import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn import metrics
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_auc_score, roc_curve


In [24]:
X = df[['age', 'gender', 'content_type_primary', 'usage_time_10_2','max_continuous_usage',
       'reels_usage_time',
        'daily_stress_level', 'overthinking_at_night',
       'sleep_latency', 'night_awakenings', 'sleep_quality', 'sleep_duration']]
y = df['insomnia_label']

In [25]:
# Split Data (80% Train, 20% Test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Train Model
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

# Predictions
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]


In [26]:

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("Classification Report:\n", classification_report(y_test, y_pred))
print("ROC-AUC Score:", roc_auc_score(y_test, y_prob))

Accuracy: 0.9565217391304348
Confusion Matrix:
 [[125   4]
 [  6  95]]
Classification Report:
               precision    recall  f1-score   support

           0       0.95      0.97      0.96       129
           1       0.96      0.94      0.95       101

    accuracy                           0.96       230
   macro avg       0.96      0.95      0.96       230
weighted avg       0.96      0.96      0.96       230

ROC-AUC Score: 0.9666896922250364


In [27]:
importance = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': model.coef_[0]
}).sort_values(by='Coefficient', ascending=False)

print(importance)

                  Feature  Coefficient
6      daily_stress_level     0.885549
9        night_awakenings     0.662045
3         usage_time_10_2     0.537704
8           sleep_latency     0.426560
7   overthinking_at_night     0.171399
2    content_type_primary     0.166712
4    max_continuous_usage     0.127534
5        reels_usage_time     0.044983
1                  gender    -0.060028
0                     age    -0.129306
11         sleep_duration    -0.167680
10          sleep_quality    -0.923914


In [28]:

age = int(input("age : "))
gender = input("gender : ")
content_type_primary = input("content_type_primary : ")
usage_time_10_2 = input("usage_time_10_2 : ")
max_continuous_usage = input("max_continuous_usage : ")
reels_usage_time = input("reels_usage_time : ")
daily_stress_level = int(input("daily_stress_level : "))
overthinking_at_night = int(input("overthinking_at_night : "))
sleep_latency = input("sleep_latency : ")
night_awakenings = int(input("night_awakenings : "))
sleep_quality = int(input("sleep_quality : "))
sleep_time = input("sleep_time : ")
wake_up_time = input("wake_up_time : ")

def convert_to_minutes(column):
    x= column.split(':')
    return int(x[0]) * 60 + int(x[1])

gender = d1[gender]
content_type_primary = d2[content_type_primary]
usage_time_10_2 = convert_to_minutes(usage_time_10_2)
reels_usage_time = convert_to_minutes(reels_usage_time)
max_continuous_usage = convert_to_minutes(max_continuous_usage)
sleep_latency = convert_to_minutes(sleep_latency)

KeyboardInterrupt: Interrupted by user

In [ ]:
#extrating sleeping duration from sleep time and wakeup time
from datetime import datetime, timedelta
def time_diff(sleep_time, wake_up_time):
    t1 = sleep_time
    t2 = wake_up_time
    start = datetime.strptime(t1, '%H:%M')
    end = datetime.strptime(t2, '%H:%M')
    delta = end - start

    if delta.days < 0:
        delta += timedelta(days=1)
    return int(delta.total_seconds() / 60)


sleep_duration = time_diff(sleep_time, wake_up_time)

In [ ]:
test = [[age, gender, content_type_primary, usage_time_10_2, max_continuous_usage, reels_usage_time, daily_stress_level, overthinking_at_night, sleep_latency,
         night_awakenings, sleep_quality, sleep_duration]]

result = model.predict(test)
print(result)
if(result == 1):
    print("You have chances of insomnia")
else:
    print("You dont have chances of insomnia")

In [30]:
import pickle
pickle.dump(model,open('insomnia_model.pkl', 'wb'))